# 🚀 Pipeline ALPR Ligero Orientado a Eventos (Vía de Lastre - Pintag)
**Arquitectura:** Crop Físico + Motion Gate 3 Estados + YOLOv8 + ByteTrack + Top-M/Top-K + FastPlateOCR + Deduplicación + Reporte Excel con Fotos Incrustadas.

> **Aceleración GPU:** Asegúrate de que el entorno de ejecución tenga una GPU asignada:
>  →  → .

In [1]:
# 1. Verificar GPU NVIDIA disponible en Colab
!nvidia-smi

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Thu Sep 17 15:53:21 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   57C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# 2. Instalar dependencias con soporte CUDA 12 para Google Colab
!pip install -q opencv-python-headless numpy openpyxl pillow pyyaml supervision ultralytics
!pip install -q fast-alpr

# Evitar conflicto de CUDA 13 en Colab fijando onnxruntime-gpu compatible con CUDA 12:
!pip uninstall -y -q onnxruntime onnxruntime-gpu
!pip install -q "onnxruntime-gpu==1.22.0"

import onnxruntime as ort
print(f"✅ ONNX Runtime: {ort.__version__}")
print(f"✅ Proveedores ONNX disponibles: {ort.get_available_providers()}")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.5/46.5 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.2/88.2 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 385.0/385.0 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 30.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.3/144.3 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.1/78.1 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 283.2/283.2 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 8.9 MB/s eta 0:00:00
✅ ONNX Runtime: 1.22.0
✅ Proveedores ONNX disponibles: ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvid

In [5]:
# 3. Montar Google Drive y localizar videos
import os
import glob

from google.colab import drive
drive.mount("/content/drive")

# Clonar repo si hace falta
if not os.path.exists("/content/PlacasVideos"):
    !git clone https://github.com/riofutabac/PlacasVideos.git /content/PlacasVideos

%cd /content/PlacasVideos

print(f"✅ Proyecto activo: {os.getcwd()}")

# ==========================================================
# CAMBIA SOLO ESTA RUTA por el nombre de tu acceso directo
# ==========================================================
VIDEOS_DIR = "/content/drive/MyDrive/Cam PL"

# Buscar todos los MP4, incluso dentro de subcarpetas
videos = glob.glob(
    os.path.join(VIDEOS_DIR, "**", "*.mp4"),
    recursive=True
)

print(f"📂 Carpeta de videos: {VIDEOS_DIR}")
print(f"📹 Videos encontrados: {len(videos)}")

for video in videos[:20]:
    print(" -", video)

if not videos:
    print("❌ No se encontraron archivos MP4.")
    print("Revisa el nombre/ruta de la carpeta.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Cloning into '/content/PlacasVideos'...
remote: Enumerating objects: 41, done.
remote: Counting objects: 100% (41/41), done.
remote: Compressing objects: 100% (33/33), done.
remote: Total 41 (delta 5), reused 40 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (41/41), 74.22 KiB | 12.37 MiB/s, done.
Resolving deltas: 100% (5/5), done.
/content/PlacasVideos
✅ Proyecto activo: /content/PlacasVideos
📂 Carpeta de videos: /content/drive/MyDrive/Cam PL
📹 Videos encontrados: 63
 - /content/drive/MyDrive/Cam PL/Camara Placas 2_20260909105651-20260909163038.mp4
 - /content/drive/MyDrive/Cam PL/Camara Placas 2_20260909105651-20260909163038(1).mp4
 - /content/drive/MyDrive/Cam PL/Camara Placas 2_20260909105651-20260909163038(2).mp4
 - /content/drive/MyDrive/Cam PL/Camara Placas 2_20260909105651-20260909163038(3).mp4
 - /content/drive/MyDrive/Cam PL/Camara Placa

In [6]:
# 4. Ejecutar el Pipeline ALPR de Extremo a Extremo en GPU
!python main.py

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
PIPELINE ALPR LIGERO ORIENTADO A EVENTOS v1.0.0
Inicio de corrida: 2026-09-17 16:04:18
Loading vehicle detector (YOLOv8n) on device='cuda'...
Loading FastALPR on providers=['CUDAExecutionProvider', 'CPUExecutionProvider'] (device='cuda')...
INFO:open_image_models.detection.core.yolo_v9.inference:Using ONNX Runtime with ['CUDAExecutionProvider', 'CPUExecutionProvider'] provider(s)
No se encontraron archivos de video con patrón 'Camara Placas 2_*.mp4'.


In [ ]:
# 5. Visualizar la Telemetría Almacenada en SQLite
import sqlite3
import pandas as pd

conn = sqlite3.connect("data/events.sqlite")
print("=== ÚLTIMA CORRIDA ===")
display(pd.read_sql("SELECT run_id, total_videos, total_events, speed_ratio, started_at, finished_at FROM processing_runs ORDER BY rowid DESC LIMIT 1", conn))

print("
=== TELEMETRÍA POR CLIP ===")
display(pd.read_sql("SELECT clip_id, wall_clock_seconds, speed_ratio, decode_seconds, vehicle_detection_seconds, processed_events FROM processed_clips ORDER BY rowid DESC LIMIT 2", conn))

print("
=== EVENTOS DETECTADOS ===")
display(pd.read_sql("SELECT event_id, datetime_str, direction, vehicle_type, plate_raw, plate_corrected, plate_status, duplicate_of FROM events ORDER BY rowid DESC LIMIT 10", conn))
conn.close()

In [ ]:
# 6. Galería de Evidencias (Fotos de Vehículos y Recortes de Placa)
import glob
from IPython.display import Image, display

veh_images = sorted(glob.glob("evidence/vehicles/*.jpg"))
print(f"Total fotos de vehículos guardadas: {len(veh_images)}")
for img_path in veh_images[:4]:
    print(f"Evidencia: {img_path}")
    display(Image(filename=img_path, width=400))

In [ ]:
# 7. Descargar el Reporte Excel de Auditoría
try:
    from google.colab import files
    if os.path.exists("reports/reporte_auditoria.xlsx"):
        files.download("reports/reporte_auditoria.xlsx")
        print("📥 Descargando reporte_auditoria.xlsx...")
except Exception as e:
    print(f"Descarga manual disponible en reports/reporte_auditoria.xlsx: {e}")